In [ ]:
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_dataset, load_metric
from transformers import MarianMTModel, MarianTokenizer
from comet import download_model, load_from_checkpoint
import torch

# -----------------------
# Setup
# -----------------------
torch.backends.mps.is_available = lambda: False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)
model.eval()

print(f"Loaded translation model: {model_name}")

# -----------------------
# Load IWSLT dataset
# -----------------------
# IWSLT17 English–French (train+test+validation available)
dataset = load_dataset("iwslt2017", "iwslt2017-en-fr")

# Use test split for evaluation
test_data = dataset["test"]

# Convert to DataFrame for easier manipulation
df = pd.DataFrame({
    "src": [ex["translation"]["en"] for ex in test_data],
    "tgt": [ex["translation"]["fr"] for ex in test_data]
})

# Optional: limit sample size for speed
df = df.head(300).reset_index(drop=True)
print(f"Loaded {len(df):,} sentence pairs from IWSLT17 (English–French).")

# -----------------------
# Translation function
# -----------------------
def translate_chunked_sentence(model, tokenizer, sentence, n, device):
    """
    Split sentence into n-word chunks, translate each, and rejoin.
    """
    words = sentence.split()
    chunks = [" ".join(words[i:i+n]) for i in range(0, len(words), n)]

    translated_chunks = []
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True).to(device)
        outputs = model.generate(**inputs, max_length=256)
        translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translated_chunks.append(translated_text)

    return " ".join(translated_chunks)

# -----------------------
# Translation + Latency
# -----------------------
context_sizes = [1]
results = {}

for n in context_sizes:
    preds, latencies = [], []
    print(f"\nTranslating with context={n} ...")

    for src in tqdm(df["src"], desc=f"context={n}"):
        start = time.time()
        pred = translate_chunked_sentence(model, tokenizer, src, n, device=device)
        end = time.time()

        preds.append(pred)
        latencies.append(end - start)

    df_context = pd.DataFrame({
        "src": df["src"],
        "tgt": df["tgt"],
        "pred": preds,
        "latency_sec": latencies
    })

    output_path = f"IWSLT_context{n}_translated.csv"
    df_context.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved translated results with latency to {output_path}")

    results[n] = df_context

# -----------------------
# Evaluation
# -----------------------
metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

final_scores = []

for n, df_context in results.items():
    print(f"\nEvaluating context={n} ...")

    src_texts = df_context["src"].tolist()
    tgt_texts = df_context["tgt"].tolist()
    preds = df_context["pred"].tolist()

    bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in tgt_texts])
    chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in tgt_texts])

    data = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(src_texts, preds, tgt_texts)]
    comet_score = comet_model.predict(data, batch_size=4, gpus=0, num_workers=0)
    comet_mean = np.mean(comet_score["system_score"])

    avg_latency = df_context["latency_sec"].mean()

    final_scores.append({
        "context": n,
        "BLEU": bleu["score"],
        "chrF": chrf["score"],
        "COMET": comet_mean,
        "Latency (sec/sentence)": avg_latency
    })

    print(
        f"Context={n}: BLEU={bleu['score']:.2f}, chrF={chrf['score']:.2f}, "
        f"COMET={comet_mean:.4f}, Latency={avg_latency:.3f}s"
    )

results_df = pd.DataFrame(final_scores)
results_df.to_csv("IWSLT_context=1_evaluation_summary_with_latency.csv", index=False)
print("\n===== Final Evaluation Summary =====")
print(results_df)
